# Step 05 — Supervised Prediction

This notebook explores the supervised classifiers trained by `pipelines/05_predict.py`.

Two families of models are trained:
- **Current-regime classifier**: predicts which regime is active *right now*
- **Forward classifiers**: predict the regime 1, 2, 4, and 8 quarters ahead

All classifiers use Random Forest by default. The input features are the same
69-column matrix used for clustering (after PCA is stripped out).

**Run `python pipelines/05_predict.py` before executing this notebook.**

## Setup & Imports

In [8]:
import sys
sys.path.insert(0, "../src")

import logging
import pickle

import pandas as pd
import numpy as np
import yaml
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import DATA_DIR, OUTPUT_DIR
from market_regime import plotting

setup_logging("INFO")
log = logging.getLogger("05_prediction")

cfg = load()

run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)

MODELS_DIR = OUTPUT_DIR / "models"
REGIMES_DIR = DATA_DIR / "regimes"
PROCESSED_DIR = DATA_DIR / "processed"

print("RunConfig:", run_cfg)
print("MODELS_DIR:", MODELS_DIR)

RunConfig: RunConfig(plots)
MODELS_DIR: /Users/glestryc/personal/github_repos/claude-scratch-work/notebooks/../outputs/models


## Load Features, Labels, and Models

In [9]:
features = None
labels = None
current_model = None
regime_names = {}

# Feature matrix
try:
    features = pd.read_parquet(PROCESSED_DIR / "features.parquet")
    print(f"Features loaded: {features.shape}")
except FileNotFoundError:
    print("ERROR: features.parquet not found. Run pipelines/02_features.py first.")

# Cluster labels
try:
    labels_df = pd.read_parquet(REGIMES_DIR / "cluster_labels.parquet")
    label_col = "balanced_cluster" if "balanced_cluster" in labels_df.columns else "cluster"
    labels = labels_df[label_col]
    print(f"Labels loaded: {len(labels)} quarters (using '{label_col}')")
except FileNotFoundError:
    print("ERROR: cluster_labels.parquet not found. Run pipelines/03_cluster.py first.")

# Regime names
for names_path in [
    REGIMES_DIR / "regime_names_suggested.yaml",
    DATA_DIR.parent / "config" / "regime_labels.yaml",
]:
    try:
        with open(names_path) as f:
            raw_names = yaml.safe_load(f)
        regime_names = {int(k): str(v) for k, v in raw_names.items()}
        print(f"Regime names loaded from {names_path}")
        break
    except FileNotFoundError:
        continue

if not regime_names and labels is not None:
    unique = sorted(labels.dropna().astype(int).unique())
    regime_names = {i: f"Regime {i}" for i in unique}
    print("Using generic regime names.")

# Current-regime classifier
try:
    with open(MODELS_DIR / "current_regime_classifier.pkl", "rb") as f:
        current_model = pickle.load(f)
    print(f"Current-regime model loaded: {type(current_model).__name__}")
except FileNotFoundError:
    print("ERROR: current_regime_classifier.pkl not found. Run pipelines/05_predict.py first.")
except Exception as exc:
    print(f"ERROR loading model: {exc}")

Features loaded: (305, 71)
Labels loaded: 304 quarters (using 'balanced_cluster')
Regime names loaded from /Users/glestryc/personal/github_repos/claude-scratch-work/notebooks/../data/regimes/regime_names_suggested.yaml
ERROR: current_regime_classifier.pkl not found. Run pipelines/05_predict.py first.


## Predict Current Regime

Run the current-regime classifier on the most recent quarter's features
and report the predicted regime with class probabilities.

In [3]:
prediction = None

if current_model is not None and features is not None:
    # Drop columns with any NaN to match training setup
    X = features.dropna(axis=1, how="any")
    latest = X.iloc[[-1]]
    latest_date = latest.index[0]

    predicted_class = int(current_model.predict(latest)[0])

    if hasattr(current_model, "predict_proba"):
        proba = current_model.predict_proba(latest)[0]
        classes = [int(c) for c in current_model.classes_]
        probabilities = dict(zip(classes, proba.tolist()))
    else:
        probabilities = {predicted_class: 1.0}

    prediction = {"regime": predicted_class, "probabilities": probabilities}

    print(f"Latest quarter: {latest_date.date()}")
    print(f"Predicted regime: {predicted_class} — {regime_names.get(predicted_class, 'Unknown')}")
    print()
    print("Class probabilities:")
    for cid in sorted(probabilities.keys()):
        name = regime_names.get(cid, f"Regime {cid}")
        bar = "#" * int(probabilities[cid] * 40)
        print(f"  {cid} {name:<35} {probabilities[cid]:6.1%}  {bar}")

## Forward Regime Probability Chart

Bar chart of predicted probabilities for the current quarter.

In [4]:
if prediction is not None:
    plotting.plot_forward_probabilities(prediction, regime_names, run_cfg)

## Feature Importance

Top-25 most important features from the current-regime Random Forest classifier.
These are the macro variables that most strongly discriminate between regimes.

In [5]:
if current_model is not None and features is not None:
    X = features.dropna(axis=1, how="any")
    plotting.plot_feature_importance(current_model, list(X.columns), run_cfg)

## Predicted vs Actual Regime Timeline

Side-by-side timeline strips showing the actual cluster assignments (from the
unsupervised step) versus what the supervised classifier would have predicted.
Good alignment confirms the classifier learned the regime structure.

In [6]:
if current_model is not None and features is not None and labels is not None:
    plotting.plot_predicted_vs_actual(features, labels, current_model, regime_names, run_cfg)

## Classification Report

Train/test split performance of the current-regime classifier.
A 20% hold-out is used for evaluation.

In [7]:
if current_model is not None and features is not None and labels is not None:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report

    X = features.dropna(axis=1, how="any")
    common = X.index.intersection(labels.dropna().index)
    X_all = X.loc[common]
    y_all = labels.loc[common].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42, shuffle=False
    )

    y_pred = current_model.predict(X_test)

    target_names = [regime_names.get(c, f"Regime {c}") for c in sorted(y_all.unique())]

    print(f"Training samples : {len(X_train)}")
    print(f"Test samples     : {len(X_test)}")
    print()
    print("Classification Report (test set):")
    print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))